In [43]:
# Transition-encoder.txt
# Translation-transformer.txt

In [44]:
import numpy as np
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, LSTM, Dense

In [45]:
data_path = [("Go.", "Va !"),
("Run!", "Cours !"),
("Run.", "Cours !"),
("Who?", "Qui ?"),
("Wow!", "Ça alors !"),
("Fire!", "Au feu !"),
("Help!", "À l'aide !"),
("Stop!", "Arrête-toi !"),
("Wait!", "Attends !"),
("Hello!", "Bonjour !"),
("I see.", "Je comprends.")]

### Data Preprocessing

In [46]:
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

for input_text, target_text in data_path:
    target_text = '\t' + target_text + '\n'
    input_texts.append(input_text)
    target_texts.append(target_text)

    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

In [47]:
input_texts

['Go.',
 'Run!',
 'Run.',
 'Who?',
 'Wow!',
 'Fire!',
 'Help!',
 'Stop!',
 'Wait!',
 'Hello!',
 'I see.']

In [48]:
target_texts

['\tVa !\n',
 '\tCours !\n',
 '\tCours !\n',
 '\tQui ?\n',
 '\tÇa alors !\n',
 '\tAu feu !\n',
 "\tÀ l'aide !\n",
 '\tArrête-toi !\n',
 '\tAttends !\n',
 '\tBonjour !\n',
 '\tJe comprends.\n']

In [49]:
input_characters=sorted(list(input_characters))
target_characters=sorted(list(target_characters))

In [50]:
input_characters

[' ',
 '!',
 '.',
 '?',
 'F',
 'G',
 'H',
 'I',
 'R',
 'S',
 'W',
 'a',
 'e',
 'h',
 'i',
 'l',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'w']

In [51]:
num_encoder_tokens=len(input_characters)
num_decoder_tokens=len(target_characters)

In [52]:
num_decoder_tokens,num_encoder_tokens

(33, 24)

In [53]:
max_encoder_seq_length=max([len(txt) for txt in input_texts])
max_decoder_seq_length=max([len(txt) for txt in target_texts])

In [54]:
max_decoder_seq_length

15

In [55]:
max_encoder_seq_length

6

In [56]:
#Create token mapping (char -> int)
input_token_index=dict([(char,i) for i,char in enumerate(input_characters)])
target_token_index=dict([(char,i) for i,char in enumerate(target_characters)])

In [57]:
input_token_index

{' ': 0,
 '!': 1,
 '.': 2,
 '?': 3,
 'F': 4,
 'G': 5,
 'H': 6,
 'I': 7,
 'R': 8,
 'S': 9,
 'W': 10,
 'a': 11,
 'e': 12,
 'h': 13,
 'i': 14,
 'l': 15,
 'n': 16,
 'o': 17,
 'p': 18,
 'r': 19,
 's': 20,
 't': 21,
 'u': 22,
 'w': 23}

In [58]:
target_token_index

{'\t': 0,
 '\n': 1,
 ' ': 2,
 '!': 3,
 "'": 4,
 '-': 5,
 '.': 6,
 '?': 7,
 'A': 8,
 'B': 9,
 'C': 10,
 'J': 11,
 'Q': 12,
 'V': 13,
 'a': 14,
 'c': 15,
 'd': 16,
 'e': 17,
 'f': 18,
 'i': 19,
 'j': 20,
 'l': 21,
 'm': 22,
 'n': 23,
 'o': 24,
 'p': 25,
 'r': 26,
 's': 27,
 't': 28,
 'u': 29,
 'À': 30,
 'Ç': 31,
 'ê': 32}

In [59]:
#Create inverse token mappping (int -> char)
reverse_input_token_index=dict([(i,char) for i,char in enumerate(input_characters)])
reverse_target_token_index=dict([(i,char) for i,char in enumerate(target_characters)])

In [60]:
reverse_input_token_index

{0: ' ',
 1: '!',
 2: '.',
 3: '?',
 4: 'F',
 5: 'G',
 6: 'H',
 7: 'I',
 8: 'R',
 9: 'S',
 10: 'W',
 11: 'a',
 12: 'e',
 13: 'h',
 14: 'i',
 15: 'l',
 16: 'n',
 17: 'o',
 18: 'p',
 19: 'r',
 20: 's',
 21: 't',
 22: 'u',
 23: 'w'}

In [61]:
reverse_target_token_index

{0: '\t',
 1: '\n',
 2: ' ',
 3: '!',
 4: "'",
 5: '-',
 6: '.',
 7: '?',
 8: 'A',
 9: 'B',
 10: 'C',
 11: 'J',
 12: 'Q',
 13: 'V',
 14: 'a',
 15: 'c',
 16: 'd',
 17: 'e',
 18: 'f',
 19: 'i',
 20: 'j',
 21: 'l',
 22: 'm',
 23: 'n',
 24: 'o',
 25: 'p',
 26: 'r',
 27: 's',
 28: 't',
 29: 'u',
 30: 'À',
 31: 'Ç',
 32: 'ê'}

In [62]:
#Generate one-hot encoded data
#Encoders Input -> (num_samples, max_len, unique_chars)
encoder_data_input=np.zeros((len(input_texts),max_encoder_seq_length,num_encoder_tokens))

In [63]:
encoder_data_input.shape

(11, 6, 24)

In [64]:
#Decoder input
decoder_data_input=np.zeros((len(input_texts),max_decoder_seq_length,num_decoder_tokens))

In [65]:
decoder_data_input.shape

(11, 15, 33)

In [66]:
decoder_target_data=np.zeros((len(input_texts),max_decoder_seq_length,num_decoder_tokens))

In [67]:
decoder_target_data.shape

(11, 15, 33)

In [68]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t,char in enumerate(input_text):
        encoder_data_input[i,t,input_token_index[char]] = 1
    encoder_data_input[i,t+1:,input_token_index[' ']] = 1

    for t,char in enumerate(target_text):
        decoder_data_input[i,t,target_token_index[char]] = 1
        if t>0:
            decoder_data_input[i,t-1,target_token_index[char]] = 1 #padding
    decoder_data_input[i,t+1:,target_token_index[' ']] = 1 #padding
    decoder_target_data[i,t:,target_token_index[' ']] = 1

In [69]:
decoder_target_data

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0.

In [70]:
encoder_data_input

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 1., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 1., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 1., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
    

In [71]:
#Build the model
encoder_inputs=Input(shape=(None,num_encoder_tokens))
encoder=LSTM(256,return_state=True)
encoder_outputs,state_h,state_c=encoder(encoder_inputs)

In [81]:
decoder_inputs=Input(shape=(None,num_decoder_tokens))
decoder_lstm=LSTM(256,return_state=True,return_sequences=True)
decoder_outputs,_,_=decoder_lstm(decoder_inputs,initial_state=[state_h,state_c])
decoder_dense=Dense(num_decoder_tokens,activation='softmax')
decoder_outputs=decoder_dense(decoder_outputs)

In [82]:
state_h

<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_2>

In [83]:
state_c

<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_3>

In [84]:
#Define model
model=Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [85]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 24)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None, 33)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    287,744 │ input_layer[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │    296,960 │ input_layer_3[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 33)  │      8,481 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 593,185 (2.26 MB)

 Trainable params: 593,185 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

In [86]:
model.compile(optimizer='rmsprop',loss='categorical_crossentropy',metrics=['accuracy'])

In [87]:
model.fit([encoder_data_input,decoder_data_input],decoder_target_data,epochs=100)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 1.2752
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.3636 - loss: 1.1564
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.3636 - loss: 1.0053
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.3636 - loss: 0.6484
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.3636 - loss: 0.0922
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3636 - loss: 0.1103
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3636 - loss: 0.0496
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.3636 - loss: 0.1437
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3636 - loss: 0.0160
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.3636 - loss: 0.1802
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3636 - loss: 0.0192
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3636 - 